# Chapter 20: Map Representations

<a href="../lite/lab/index.html?path=ch20_map_representations.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

What is a map? To Google Maps, it is road segments and GPS coordinates. To a vacuum robot,
it is a grid of "clean" and "not yet clean" cells. To a self driving car, it is a centimeter
precise 3D model of every lane marking and curb. The representation you choose for your
map determines what your robot can and cannot do.

This chapter explores the main map representations used in robotics: **landmarks**, **occupancy
grids**, **sparse** vs **dense** maps, and **metric** vs **topological** maps.

## 20.1 Landmarks

A **landmark map** stores a list of distinct features: their positions (and possibly descriptors).
This is the simplest representation and the one used by EKF-SLAM.

$$\mathcal{M} = \{(\mathbf{l}_1, d_1), (\mathbf{l}_2, d_2), \ldots, (\mathbf{l}_N, d_N)\}$$

where $\mathbf{l}_i$ is the position and $d_i$ is a descriptor (e.g., color, shape).

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
n_landmarks = 15
room_size = 20.0
# ──────────────────────────────────────────────────────────────────────────────

landmarks = np.random.uniform(1, room_size - 1, (n_landmarks, 2))
types = np.random.choice(['tree', 'pole', 'sign'], n_landmarks)
colors = {'tree': 'forestgreen', 'pole': 'steelblue', 'sign': 'tomato'}

fig, ax = plt.subplots(figsize=(8, 8))
for t in ['tree', 'pole', 'sign']:
    mask = types == t
    ax.scatter(landmarks[mask, 0], landmarks[mask, 1], c=colors[t], s=100,
              marker={'tree': '^', 'pole': 's', 'sign': 'D'}[t], label=t, zorder=5)
ax.set_xlim(0, room_size); ax.set_ylim(0, room_size)
ax.set_aspect('equal')
ax.set_title(f"Landmark map: {n_landmarks} features", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Map storage: {n_landmarks} × (2 position + 1 type) = {n_landmarks * 3} values")

## 20.2 Occupancy Grids

An **occupancy grid** divides the world into cells. Each cell stores the probability
that it is occupied. This is the standard representation for 2D navigation.

The **log-odds** representation is numerically stable:

$$l_t = l_{t-1} + \log\frac{p(m \mid z_t)}{1 - p(m \mid z_t)} - l_0$$

where $l_0$ is the prior log-odds.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
grid_size = 50          # cells per side
cell_size = 0.4         # meters per cell
robot_pos = np.array([10.0, 10.0])   # robot position (meters)
n_beams = 36            # number of LiDAR beams
max_range = 8.0
# ──────────────────────────────────────────────────────────────────────────────

# Create a room with walls
room_w = grid_size * cell_size
log_odds = np.zeros((grid_size, grid_size))  # prior: 0 = unknown

# Define walls
wall_cells = set()
for i in range(grid_size):
    wall_cells.add((0, i)); wall_cells.add((grid_size-1, i))
    wall_cells.add((i, 0)); wall_cells.add((i, grid_size-1))
# Add an internal wall
for i in range(15, 35):
    wall_cells.add((i, 25))

# Simulate LiDAR and update occupancy grid
angles = np.linspace(0, 2*np.pi, n_beams, endpoint=False)
for angle in angles:
    for r in np.arange(0, max_range, cell_size * 0.5):
        px = robot_pos[0] + r * np.cos(angle)
        py = robot_pos[1] + r * np.sin(angle)
        ci = int(px / cell_size)
        cj = int(py / cell_size)
        if 0 <= ci < grid_size and 0 <= cj < grid_size:
            if (ci, cj) in wall_cells:
                log_odds[cj, ci] += 0.7  # occupied
                break
            else:
                log_odds[cj, ci] -= 0.4  # free

# Convert log-odds to probability
prob = 1 - 1 / (1 + np.exp(log_odds))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.imshow(log_odds, cmap='RdBu_r', origin='lower', extent=[0, room_w, 0, room_w])
ax.plot(*robot_pos, 'ko', ms=8, zorder=5)
ax.set_title("Log-odds grid", fontsize=13)

ax = axes[1]
ax.imshow(prob, cmap='gray_r', origin='lower', extent=[0, room_w, 0, room_w], vmin=0, vmax=1)
ax.plot(*robot_pos, 'ro', ms=8, zorder=5)
ax.set_title("Occupancy probability (white=free, black=occupied)", fontsize=13)

plt.tight_layout()
plt.show()

## 20.3 Sparse vs Dense

| Property | Sparse (landmarks) | Dense (occupancy grid) |
|----------|:---:|:---:|
| **Storage** | Low ($O(N)$ landmarks) | High ($O(W \times H)$ cells) |
| **Path planning** | Needs separate representation | Direct (A*, wavefront) |
| **Data association** | Feature matching | Scan matching |
| **Typical use** | EKF-SLAM, visual SLAM | Navigation, obstacle avoidance |

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
n_sparse = 20        # number of landmarks
grid_res = 100       # grid resolution for dense map
# ──────────────────────────────────────────────────────────────────────────────

np.random.seed(42)
sparse_map = np.random.uniform(1, 19, (n_sparse, 2))

dense_map = np.zeros((grid_res, grid_res))
# Create walls and obstacles in dense map
dense_map[0, :] = dense_map[-1, :] = dense_map[:, 0] = dense_map[:, -1] = 1
dense_map[30:70, 50] = 1  # internal wall
dense_map[40, 20:60] = 1  # another wall

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(sparse_map[:, 0], sparse_map[:, 1], c='steelblue', s=80, marker='^', zorder=5)
ax.set_xlim(0, 20); ax.set_ylim(0, 20); ax.set_aspect('equal')
ax.set_title(f"Sparse map: {n_sparse} landmarks\nStorage: {n_sparse * 2} floats", fontsize=12)

ax = axes[1]
ax.imshow(dense_map, cmap='gray_r', origin='lower', extent=[0, 20, 0, 20])
ax.set_title(f"Dense map: {grid_res}×{grid_res} grid\nStorage: {grid_res**2} floats", fontsize=12)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## 20.4 Metric vs Topological

A **metric map** stores exact positions and distances. A **topological map** stores
only the connectivity: which places are connected to which, and roughly how far apart.

Topological maps are compact and sufficient for high level planning ("go from the kitchen
to the bedroom"), but cannot guide precise navigation ("drive exactly 2.3 meters forward").

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
# Define a simple topological map of a house
places = {'Kitchen': (2, 3), 'Living': (5, 3), 'Bedroom': (8, 3),
          'Hallway': (5, 6), 'Bathroom': (2, 6), 'Garage': (8, 6)}
edges = [('Kitchen', 'Living'), ('Living', 'Bedroom'), ('Living', 'Hallway'),
         ('Hallway', 'Bathroom'), ('Hallway', 'Bedroom'), ('Kitchen', 'Bathroom'),
         ('Bedroom', 'Garage')]
# ──────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Metric
ax = axes[0]
for name, pos in places.items():
    ax.plot(*pos, 'o', color='steelblue', ms=15, zorder=5)
    ax.annotate(name, xy=pos, xytext=(0, -20), textcoords='offset points',
                ha='center', fontsize=10, fontweight='bold')
for a, b in edges:
    pa, pb = places[a], places[b]
    dist = np.sqrt((pa[0]-pb[0])**2 + (pa[1]-pb[1])**2)
    ax.plot([pa[0], pb[0]], [pa[1], pb[1]], 'gray', lw=2)
    mx, my = (pa[0]+pb[0])/2, (pa[1]+pb[1])/2
    ax.text(mx, my+0.3, f'{dist:.1f}m', fontsize=8, ha='center', color='gray')
ax.set_title("Metric map (exact positions)", fontsize=13)
ax.set_xlim(0, 10); ax.set_ylim(1, 8); ax.set_aspect('equal')
ax.axis('off')

# Topological
ax = axes[1]
y_positions = {'Kitchen': 1, 'Bathroom': 1, 'Living': 2, 'Hallway': 2, 'Bedroom': 3, 'Garage': 3}
x_positions = {'Kitchen': 1, 'Bathroom': 3, 'Living': 2, 'Hallway': 4, 'Bedroom': 2, 'Garage': 4}
for name in places:
    ax.plot(x_positions[name], y_positions[name], 'o', color='tomato', ms=15, zorder=5)
    ax.annotate(name, xy=(x_positions[name], y_positions[name]),
                xytext=(0, -20), textcoords='offset points', ha='center', fontsize=10, fontweight='bold')
for a, b in edges:
    ax.plot([x_positions[a], x_positions[b]], [y_positions[a], y_positions[b]], 'gray', lw=2)
ax.set_title("Topological map (connectivity only)", fontsize=13)
ax.set_xlim(0, 5); ax.set_ylim(0, 4)
ax.axis('off')

plt.tight_layout()
plt.show()

**Key observations:**
- **Landmark maps** are compact and work well with EKF-SLAM but cannot represent free space.
- **Occupancy grids** represent free space explicitly, enabling path planning, but scale poorly to 3D.
- **Topological maps** are the most compact and support high level reasoning but lack metric precision.
- Modern SLAM systems often use **hybrid** representations: a topological backbone with local metric maps at each node.

---

## Exercises

### Exercise 20.1: Build an occupancy grid

Implement an occupancy grid for a 10m × 10m room with 0.1m resolution (100×100 grid).
Place walls on all four sides and one internal wall. Simulate a LiDAR from the center and
update the grid. Display the result.

In [ ]:
# Your code here
# Initialize log_odds = np.zeros((100, 100))
# For each beam: trace ray, mark free cells with -0.4, mark hit cell with +0.7

### Exercise 20.2: Compare storage costs

For a 50m × 50m environment, compute the storage (number of floats) needed for:
(a) Occupancy grid at 5cm resolution
(b) 200 point landmarks
(c) A topological map with 10 nodes and 15 edges
Which is most efficient? When would you choose each?

In [ ]:
# Your code here

### Exercise 20.3: Occupancy grid from multiple scans (challenge)

Simulate a robot driving through a room taking LiDAR scans at 5 different positions.
Fuse all scans into a single occupancy grid using log-odds updates.
Show the grid after each scan is added. The map should become more defined with each scan.

In [ ]:
# Your code here
# For each robot position:
#   Generate LiDAR scan
#   Update log_odds grid
#   Plot intermediate result